In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
def split_et_scale_proprement_sans_stratify(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test


def charger_immobilier():
    data = fetch_california_housing()
    X = data.data
    y = data.target

    print(
        f"California Housing : {X.shape} variables, "
        "cible = prix médian en centaines de milliers de $"
    )
    print("\n Features :")
    print(data.feature_names)

    return X, y

def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)

    y_pred = modele.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)

    return {
        "r2": r2,
        "mae": mae,
        "rmse": rmse
    }

def checkpoint_qualite(X, y, case = "normal"):
    if case == "limite":
        X_small = X[:100]
        y_small = y[:100]
        return X_small, y_small
    
    return X, y

In [ ]:
print("======Phase A : Prédire les prix immobiliers (régression)=========")

X, y = charger_immobilier()
lr = make_pipeline(StandardScaler(), LinearRegression())
rf = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42))
print("======CAS NORMAL=========")
X_checked, y_checked = checkpoint_qualite(X, y)
X_train_scaled, X_test_scaled, y_train, y_test = split_et_scale_proprement_sans_stratify(X_checked, y_checked)
resultats_lr = evaluer_regression(
        lr,
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test
    )

print(
    f"\n LinearRegression : "
    f"R2={resultats_lr['r2']:.2f} "
    f"MAE={resultats_lr['mae']:.2f} "
    f"RMSE={resultats_lr['rmse']:.2f}"
)
resultats_rf = evaluer_regression(
    rf,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)
print(
    f"\n RandomForest : "
    f"R2={resultats_rf['r2']:.2f} "
    f"MAE={resultats_rf['mae']:.2f} "
    f"RMSE={resultats_rf['rmse']:.2f}"
)


print("======CAS LIMITE=========")
X_checked, y_checked = checkpoint_qualite(X, y, "limite")
X_train_scaled, X_test_scaled, y_train, y_test = split_et_scale_proprement_sans_stratify(X_checked, y_checked)
resultats_lr = evaluer_regression(
        lr,
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test
    )

print(
    f"\n LinearRegression : "
    f"R2={resultats_lr['r2']:.2f} "
    f"MAE={resultats_lr['mae']:.2f} "
    f"RMSE={resultats_lr['rmse']:.2f}"
)
resultats_rf = evaluer_regression(
    rf,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)
print(
    f"\n RandomForest : "
    f"R2={resultats_rf['r2']:.2f} "
    f"MAE={resultats_rf['mae']:.2f} "
    f"RMSE={resultats_rf['rmse']:.2f}"
)

print("\n ======CAS ADVERSARIAL=========")
quartier_fictif = np.array([[
    0.0,
    20.0,
    5.0,
    1.0,
    9000.0,
    4.0,
    34.0,
    -118.0
]])

prix_lr = lr.predict(quartier_fictif)[0]
prix_rf = rf.predict(quartier_fictif)[0]
print(f"LinearRegression prédit : {prix_lr:.1f} $")
print(f"RandomForest     prédit : {prix_rf:.1f} $")

In [ ]:
import pandas as pd

In [ ]:
def charger_airbnb(url_csv):
    df = pd.read_csv(url_csv)
    colonnes_pertinentes = [
        "price",
        "minimum_nights",
        "number_of_reviews",
        "availability_365",
        "calculated_host_listings_count"
    ]

    cols = [c for c in colonnes_pertinentes if c in df.columns]
    df = df[cols].copy()
    for c in cols:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna()
    print(f"Listings chargés : {df.shape[0]} lignes, {df.shape[1]} colonnes numériques retenues")

    return df


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def choisir_k(X_scaled, k_range=range(2, 9)):
    print("\n=====Choix du K=====")

    results = []

    for k in k_range:
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X_scaled)

        inertia = kmeans.inertia_
        silhouette = silhouette_score(X_scaled, labels)

        results.append((k, inertia, silhouette))

        print(f"k={k} : inertie={inertia:.0f} silhouette={silhouette:.2f}")

    best_k = max(results, key=lambda x: x[2])[0]

    print(f"\nSegment retenu : k={best_k}")

    return best_k


def pipeline_airbnb(df):
    X = df.values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    k = choisir_k(X_scaled)

    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    labels = model.fit_predict(X_scaled)

    return X_scaled, labels, model


def interpretation_segments(row):

    if row["price"] > 150 and row["availability_365"] < 200:
        return "Premium / haute demande"

    elif row["number_of_reviews"] > 50:
        return "Populaire / très commenté"

    else:
        return "Budget / faible activité"

In [ ]:
URL_AIRBNB = "https://data.insideairbnb.com/canada/on/ottawa/2025-09-22/visualisations/listings.csv"

print("\n ======Phase B : Segmenter les clients d'AirBnB (non supervisé)=========")
df = charger_airbnb(URL_AIRBNB)
print("======CAS NORMAL=========")
X_scaled, labels, model = pipeline_airbnb(df)
df_profile = df.copy()
df_profile["cluster"] = labels
profil = df_profile.groupby("cluster").mean()
profil_normalise = (
    profil - profil.mean()
) / profil.std()

print(profil_normalise)
df_profile["segment"] = df_profile.apply(interpretation_segments, axis=1)
print(df_profile["segment"].value_counts())

print("======CAS LIMITE=========")
kmeans = KMeans(n_clusters=3, n_init=10)
labels = kmeans.fit_predict(df.values)

df_profile = df.copy()
df_profile["cluster"] = labels
profil = df_profile.groupby("cluster").mean()
profil_normalise = (
    profil - profil.mean()
) / profil.std()

print(profil_normalise)

print("======CAS ADVERSARIAL=========")
df_bad = df.copy()

df_bad.loc[0, "price"] = 100000
scaler_bad = StandardScaler()
X_scaled = scaler_bad.fit_transform(df_bad)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
km_bad = kmeans.fit(X_scaled)

clusters = pd.DataFrame(
    scaler_bad.inverse_transform(km_bad.cluster_centers_),
    columns=df.columns
)
print(clusters)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

In [ ]:
def vectoriser_textes(messages, vectorizer=None):
    if vectorizer is None:
        vectorizer = TfidfVectorizer()

        X = vectorizer.fit_transform(messages)

    else:
        X = vectorizer.transform(messages)

    return X, vectorizer

def evaluer_spam(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)

    y_pred = modele.predict(X_test)

    print(classification_report(y_test, y_pred, target_names=["normal", "spam"]))

    return y_pred

In [ ]:
print("\n======Phase C : Courriel vs spam (texte)=========")
# chargement du dataset
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    names=["label", "message"]
)

df["label_raw"] = df["label"]
# labels numériques

df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

print(df)


X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["message"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

X_train, vectorizer = vectoriser_textes(X_train_text)

X_test, _ = vectoriser_textes(
    X_test_text,
    vectorizer
)

print("\n=====Happy path case=====")
print("\n=====MultinomialNB=====")

evaluer_spam(
    MultinomialNB(),
    X_train,
    X_test,
    y_train,
    y_test
)

print("\n=====Logistic Regression=====")

evaluer_spam(
    LogisticRegression(max_iter=1000),
    X_train,
    X_test,
    y_train,
    y_test
)

print("\n=====Edge case=====")
msg = [""]

X_msg, _ = vectoriser_textes(
    msg,
    vectorizer
)
print("\n=====MultinomialNB=====")

modeleMB = MultinomialNB()
modeleMB.fit(X_train, y_train)

prediction = modeleMB.predict(X_msg)
proba = modeleMB.predict_proba(X_msg)

print("proba : ", proba)

print(
    "Prédiction :",
    "spam" if prediction[0] == 1 else "normal"
)


print("\n=====Logistic Regression=====")

modeleLR = LogisticRegression()
modeleLR.fit(X_train, y_train)

prediction = modeleLR.predict(X_msg)
proba = modeleLR.predict_proba(X_msg)

print("proba : ", proba)
print(
    "Prédiction :",
    "spam" if prediction[0] == 1 else "normal"
)

print("\n=====Adversarial case=====")

spam_camoufle = [
    "Salut, ton colis t'attend, confirme ici"
]
X_spam_camoufle, _ = vectoriser_textes(
    spam_camoufle,
    vectorizer
)
print("\n=====MultinomialNB=====")

modeleMB = MultinomialNB()
modeleMB.fit(X_train, y_train)

prediction = modeleMB.predict(X_spam_camoufle)
proba = modeleMB.predict_proba(X_msg)

print("proba : ", proba)
print(
    "Prédiction :",
    "spam" if prediction[0] == 1 else "normal"
)

print("\n=====Logistic Regression=====")

modeleLR = LogisticRegression()
modeleLR.fit(X_train, y_train)

prediction = modeleLR.predict(X_spam_camoufle)

proba = modeleLR.predict_proba(X_msg)

print("proba : ", proba)
print(
    "Prédiction :",
    "spam" if prediction[0] == 1 else "normal"
)